# Equity models: calibrate and price

Result-driven: calibrate the equity models on one ticker, then price a put and a digital with each of them, with the
greeks to the market inputs, against Black-Scholes. How the models work: `local_vol.ipynb`, `heston.ipynb`,
`heston_slv.ipynb`.

In [ ]:
TICKER = "TICKER1"
STRIKE = 0.80            # strike / spot
EXPIRY_DAYS = 365
MC = dict(n_paths=50_000, n_steps=100, seed=1)

In [ ]:
import os
import time
from datetime import timedelta
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
import tensorquant as tq

mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")       # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date
eq = mkt.get_equity(TICKER)
EXPIRY = mkt.evaluation_date + timedelta(days=EXPIRY_DAYS)
K = round(STRIKE * float(eq.spot), 2)
print(f"{TICKER}  {mkt.evaluation_date}  spot {float(eq.spot):.2f}  K {K:.2f}  F(T) {float(eq.forward(EXPIRY)):.2f}")

## 1. Calibration

`Model.calibrate(equity)` fits once; `model.process(equity)` binds it to today's market (local vol grid, forwards).

In [ ]:
MODELS = {"local vol": tq.LocalVolatilityModel, "Heston": tq.HestonModel,
          "displaced diffusion": tq.DisplacedDiffusionModel, "Heston SLV": tq.HestonSLVModel}
models, timing = {}, []
for name, cls in MODELS.items():
    t0 = time.perf_counter()
    models[name] = cls.calibrate(eq)
    t1 = time.perf_counter()
    models[name].process(eq)
    timing.append({"model": name, "calibrate [s]": t1 - t0, "process [s]": time.perf_counter() - t1,
                   "parameters": {k: np.round(v.numpy(), 4).tolist() for k, v in models[name].parameters.items()}})
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(timing).set_index("model").round(3)

### Calibration report

Every option of the vol surface (out of the money: puts below the forward, calls above) priced with Black at the
market vol and with the model, then the model price inverted back to a Black vol. Heston and displaced diffusion by
their closed forms; local vol and SLV by Monte Carlo (one simulation per model observed at all the maturities,
`SURFACE_MC`: at least 25 steps between maturities, otherwise the 1M slice has 4 steps and the discretisation bias
dominates), so their differences include the MC noise: on the 1M wings the prices are a few hundredths of a % of spot
and the noise reads as vol points. Price differences in % of spot, vol differences in vol points.

In [ ]:
from scipy.optimize import brentq
from scipy.stats import norm

SURFACE_MC = dict(n_paths=50_000, steps_per_year=100, min_steps=25, seed=11)   # >= 25 steps between maturities
vs, S0 = eq.vol_surface, float(eq.spot)
T_NODES, K_NODES = np.asarray(vs.maturity, float), np.asarray(vs.strike, float)
FWD = np.array([float(eq.forward(t)) for t in T_NODES])
DF = np.array([float(eq.discount_curve.discount(t)) for t in T_NODES])
PHI = np.where(K_NODES[None, :] >= FWD[:, None], 1.0, -1.0)                  # OTM: put below the forward
MARKET_VOL = vs.volatility_matrix.numpy()
GRID = dict(index=pd.Index(np.round(T_NODES, 2), name="T"),
            columns=pd.Index(np.round(100 * K_NODES / S0).astype(int), name="K/S0 %"))


def black(F, K, T, vol, phi, df):
    sd = vol * np.sqrt(T)
    d1 = np.log(F / K) / sd + 0.5 * sd
    return df * phi * (F * norm.cdf(phi * d1) - K * norm.cdf(phi * (d1 - sd)))


def implied_vol(price, F, K, T, phi, df):
    if not price > df * max(phi * (F - K), 0.0):
        return np.nan
    return brentq(lambda v: black(F, K, T, v, phi, df) - price, 1e-4, 5.0)


def surface_prices(process):
    if hasattr(process, "vanilla_price"):                                    # closed form
        return np.array([[float(process.vanilla_price(k, t, PHI[i, j])) for j, k in enumerate(K_NODES)]
                         for i, t in enumerate(T_NODES)])
    edges = np.concatenate([[0.0], T_NODES])
    times = np.concatenate([np.linspace(a, b, max(SURFACE_MC["min_steps"], int(np.ceil(SURFACE_MC["steps_per_year"] * (b - a)))) + 1)[1:]
                            for a, b in zip(edges[:-1], edges[1:])])
    k = process.noise_size()
    dw = tf.random.stateless_normal([SURFACE_MC["n_paths"], len(times)] + ([k] if k > 1 else []),
                                    seed=[SURFACE_MC["seed"], 0], dtype=tf.float64)
    s_t = process.evolve(tf.constant(times, tf.float64), dw, observation_times=list(T_NODES)).numpy()
    payoff = np.maximum(PHI[None] * (s_t[:, :, None] - K_NODES[None, None, :]), 0.0)
    return DF[:, None] * payoff.mean(axis=0)


MARKET_PRICE = black(FWD[:, None], K_NODES[None, :], T_NODES[:, None], MARKET_VOL, PHI, DF[:, None])
fits, summary_rows = {}, []
for name, model in models.items():
    t0 = time.perf_counter()
    price = surface_prices(model.process(eq))
    vol = np.array([[implied_vol(price[i, j], FWD[i], K_NODES[j], T_NODES[i], PHI[i, j], DF[i])
                     for j in range(len(K_NODES))] for i in range(len(T_NODES))])
    fits[name] = (pd.DataFrame(100 * (price - MARKET_PRICE) / S0, **GRID), pd.DataFrame(100 * (vol - MARKET_VOL), **GRID))
    v = fits[name][1].to_numpy()
    summary_rows.append({"model": name, "max |price diff| [% S0]": np.nanmax(np.abs(fits[name][0].to_numpy())),
                         "rms vol diff [vol pt]": np.sqrt(np.nanmean(v ** 2)), "max |vol diff| [vol pt]": np.nanmax(np.abs(v)),
                         "repricing [s]": time.perf_counter() - t0})
pd.DataFrame(summary_rows).set_index("model").round(2)

In [ ]:
def show_fit(name):
    price, vol = fits[name]
    print(f"{name}: model - Black price, % of spot")
    display(price.round(2))
    print(f"{name}: model - market implied vol, vol points")
    display(vol.round(2))


for name in models:
    show_fit(name)

## 2. Put 80%, 1Y

A trade is a `Product`; a pricer prices it under a `GradientTape`, and one gradient to `equity.risk_factors()` gives
all the greeks. For a calibrated model the greeks go through the calibration: `par_greeks` adds the model greeks
mapped by the stored Jacobian. Delta per unit of spot, vega per vol point (sum over the surface nodes), rho per bp
(sum over the curve pillars).

In [ ]:
SPOT, VOL = (next(k for k in eq.risk_factors() if k.startswith("EQ:") and k.endswith(s)) for s in (":SPOT", ":VOL"))


def summary(name, price, tape, model=None, se=0.0, seconds=None):
    params = model.parameters if model is not None else {}
    g = tape.gradient(price, {"model": params, "market": eq.risk_factors()})
    market = {k: [None if x is None else x.numpy() for x in v] for k, v in g["market"].items()}
    greeks = tq.par_greeks(model, {k: v.numpy() for k, v in g["model"].items()}, market, equity=eq) if model else \
        {k: [np.zeros(()) if x is None else x for x in v] for k, v in market.items()}
    rho = sum(np.sum(x) for k, v in greeks.items() if k.startswith("IR:") for x in v)
    return {"model": name, "price": float(price), "s.e.": float(se or 0.0),
            "delta": float(np.sum(greeks[SPOT][0])), "vega (1 vol pt)": float(np.sum(greeks[VOL][0])) / 100,
            "rho (1 bp)": float(rho) * 1e-4, "time [s]": seconds}, greeks


def black_scholes(product, name="Black-Scholes", **kwargs):
    pricer, t0 = tq.BlackScholesPricer(**kwargs), time.perf_counter()
    pricer.price(product, mkt, autodiff=True)
    return summary(name, product.price, pricer.tape, seconds=time.perf_counter() - t0)


def price_all(make_product, benchmarks):
    rows, vegas = [], {}
    for row, greeks in benchmarks:
        rows.append(row); vegas[row["model"]] = greeks[VOL][0]
    for name, model in models.items():
        product, pricer = make_product(), tq.VanillaMCPricer(model=model, **MC)
        t0 = time.perf_counter()
        pricer.price(product, mkt, autodiff=True)
        row, greeks = summary(name, product.price, pricer.tape, model, product.standard_error,
                              time.perf_counter() - t0)
        rows.append(row); vegas[name] = greeks[VOL][0]
    df = pd.DataFrame(rows).set_index("model")
    reference = rows[0]["model"]
    df.insert(2, f"vs {reference}", df["price"] - df.loc[reference, "price"])
    df.insert(df.columns.get_loc("vega (1 vol pt)") + 1, "vega vs ref",
              df["vega (1 vol pt)"] - df.loc[reference, "vega (1 vol pt)"])
    return df, vegas

In [ ]:
def put(strike=K):
    return tq.VanillaOption(eq.currency, start_date=mkt.evaluation_date, end_date=EXPIRY,
                            option_type=tq.OptionType.Put, strike=strike, underlying=TICKER)


put_table, put_vegas = price_all(put, [black_scholes(put())])
put_table.round(4)

How each model spreads the vega over the surface nodes (per vol point, same colour scale for all; cells below 0.005
left blank). For a vanilla the risk reference is the Black-Scholes vega on its own node.

- **local vol**: on the node and its neighbours (the local vol is built from the interpolated surface); AAD = bump.
- **Heston**: the vega is mapped through the whole-surface calibration of five parameters, so it lands on every
  maturity (negative at 5Y): right for the model, not a vanilla's risk (`ROADMAP.md`: IMI-style vega by structure).
- **Heston SLV**: like local vol (on the node and its neighbours): the vega through the calibrated `E[v | X]` table is
  mapped by the adjoint of its calibration (`par_greeks(..., equity=eq)`); without it part of the vega would land on
  other strikes of the same expiry. Price on the bias-corrected table, greeks on the smooth 41-node one (smoothing
  for risk, `heston_slv.ipynb`).

In [ ]:
import matplotlib.pyplot as plt


def vega_maps(vegas, title):
    maps = {name: v / 100 for name, v in vegas.items()}
    scale = max(np.abs(v).max() for v in maps.values())
    fig, axes = plt.subplots(1, len(maps), figsize=(3.3 * len(maps), 3.4), sharey=True, constrained_layout=True)
    for ax, (name, v) in zip(axes, maps.items()):
        im = ax.imshow(v, cmap="RdBu_r", vmin=-scale, vmax=scale, aspect="auto")
        for (i, j), x in np.ndenumerate(v):
            if abs(x) >= 0.005:
                ax.text(j, i, f"{x:.2f}", ha="center", va="center", fontsize=6.5,
                        color="white" if abs(x) > 0.6 * scale else "black")
        ax.set_title(name, fontsize=10)
        ax.set_xticks(range(len(K_NODES)), GRID["columns"], fontsize=7)
        ax.set_xlabel("K / S0 %", fontsize=8)
    axes[0].set_yticks(range(len(T_NODES)), GRID["index"], fontsize=7)
    axes[0].set_ylabel("T (years)", fontsize=8)
    fig.colorbar(im, ax=axes, shrink=0.85, label="vega per vol point")
    fig.suptitle(title)


vega_maps(put_vegas, f"Put {STRIKE:.0%}, {EXPIRY_DAYS}d: vega per surface node")

## 3. Digital put 80%, 1Y (cash-or-nothing, pays 100)

Benchmark: the **put spread** on the smile, `BlackScholesPricer(digital_spread=h)`:
`100 * (P(K + hK) - P(K - hK)) / 2hK` with each put at its own vol (a digital put is `dP/dK`), the market price of
the digital, no model. Black-Scholes at the vol of the strike (the default) misses the skew term,
`digital = BS digital + vega * dsigma/dK`, shown for reference. Monte Carlo smooths the indicator (sigmoid, 1% of the
strike) so the greeks exist.

In [ ]:
def digital_put():
    return tq.DigitalOption(eq.currency, mkt.evaluation_date, EXPIRY, tq.OptionType.Put, K,
                            payoff_type=tq.PayoffType.CashOrNothing, cash_amount=100.0, underlying=TICKER)


benchmarks = [black_scholes(digital_put(), "put spread (smile)", digital_spread=0.005), black_scholes(digital_put())]
digital_table, digital_vegas = price_all(digital_put, benchmarks)
digital_table.round(4)

In [ ]:
vega_maps(digital_vegas, f"Digital put {STRIKE:.0%}, {EXPIRY_DAYS}d: vega per surface node")